In [1]:
#----Graph-based anomaly detection---Can we automatically flag unusual or contradictory relationships in a patient's oncology history?
"""
Examples of potential anomalies include:
- A treatment stop date before its start date.
- A response assessment dated before the treatment it supposedly evaluates.
- A relationship connecting events that have inconsistent patient identifiers.
- A missing source for a critical clinical event.
Not every unusual sequence is an error. An anomaly detector should flag records for investigation rather than automatically 
correct them.

Outcome of below program - 
E3: The recorded treatment stop predates the treatment start. This could indicate a date error or a missing earlier treatment course.
E4: The assessment lacks source evidence and is dated before the recorded treatment start. It may be a baseline assessment rather 
than a post-treatment response assessment. The last example is particularly important: an assessment before treatment is not 
necessarily wrong. It becomes suspicious only if it is classified as a response to that later treatment.
"""
# Illustrative patient events.# In production, each event should have a patient ID, # treatment-course ID, date, and source reference.
events = [
    {
        "id"        : "E1",
        "patient"   : "P101",
        "type"      : "Biomarker_Result",
        "date"      : "2026-01-10",
        "source"    : "Molecular_Report_12"
    },
    {
        "id"        : "E2",
        "patient"   : "P101",
        "type"      : "Treatment_Start",
        "date"      : "2026-01-20",
        "source"    : "Prescription_22"
    },
    {
        "id"        : "E3",
        "patient"   : "P101",
        "type"      : "Treatment_Stop",
        "date"      : "2026-01-05",
        "source"    : "Oncology_Note_18"
    },
    {
        "id"        : "E4",
        "patient"   : "P101",
        "type"      : "Response_Assessment",
        "date"      : "2026-01-15",
        "source"    : None
    }
]
def detect_anomalies(events):
    anomalies = []
    by_type = {event["type"]: event for event in events}
    for event in events:
        if not event["source"]:                                     # Rule 1: Missing source evidence.
            anomalies.append((
                event["id"],
                "Missing source evidence"
            ))
        if event["type"] == "Treatment_Stop":                       # Rule 2: Treatment stop before treatment start.
            start = by_type.get("Treatment_Start")
            if start and event["date"] < start["date"]:
                anomalies.append((
                    event["id"],
                    "Stop date precedes start date"
                ))        
        if event["type"] == "Response_Assessment":                  # Rule 3: Response assessment before treatment start.
            start = by_type.get("Treatment_Start")
            if start and event["date"] < start["date"]:
                anomalies.append((
                    event["id"],
                    "Response assessment precedes treatment start"
                ))
    return anomalies
results = detect_anomalies(events)
print("Potential anomalies:")
for event_id, reason in results:
    print(event_id, "->", reason)

Potential anomalies:
E3 -> Stop date precedes start date
E4 -> Missing source evidence
E4 -> Response assessment precedes treatment start
